# 清理 MIDI：复制到新文件夹并统一 Channel 1



In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import mido
from collections import Counter, defaultdict

## 1. 路径配置

确认你的项目结构中已经有：

```text
D:\organ-amt-generalization\scripts\clean_midi_channel.py
```

如果还没有，把之前生成的 `clean_midi_channel.py` 放到这个位置。

In [2]:
PROJECT_ROOT = Path(r"D:\organ-amt-generalization")

SCRIPT_PATH = PROJECT_ROOT / "scripts" / "clean_midi_channel.py"
CONFIG_PATH = PROJECT_ROOT / "configs" / "clean_midi_channel.yaml"

INPUT_MIDI_DIR = PROJECT_ROOT / "data" / "raw" / "organ" / "BWV545" / "midi"
OUTPUT_MIDI_DIR = PROJECT_ROOT / "data" / "processed" / "clearmidi"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("INPUT_MIDI_DIR:", INPUT_MIDI_DIR)
print("OUTPUT_MIDI_DIR:", OUTPUT_MIDI_DIR)

assert PROJECT_ROOT.exists(), f"项目根目录不存在: {PROJECT_ROOT}"
assert SCRIPT_PATH.exists(), f"清理脚本不存在，请先放到: {SCRIPT_PATH}"
assert INPUT_MIDI_DIR.exists(), f"输入 MIDI 文件夹不存在: {INPUT_MIDI_DIR}"

PROJECT_ROOT: D:\organ-amt-generalization
SCRIPT_PATH: D:\organ-amt-generalization\scripts\clean_midi_channel.py
CONFIG_PATH: D:\organ-amt-generalization\configs\clean_midi_channel.yaml
INPUT_MIDI_DIR: D:\organ-amt-generalization\data\raw\organ\BWV545\midi
OUTPUT_MIDI_DIR: D:\organ-amt-generalization\data\processed\clearmidi


## 2. 写入配置文件

这里会把配置文件写到：

```text
D:\organ-amt-generalization\configs\clean_midi_channel.yaml
```

注意：配置中使用 `/` 形式的路径，避免 YAML 把 Windows 反斜杠当成转义字符。

In [3]:
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_MIDI_DIR.mkdir(parents=True, exist_ok=True)

config_text = f'''# clean_midi_channel.yaml

# 输入 MIDI 文件夹：只读取，不修改原文件
input_midi_dir: "{INPUT_MIDI_DIR.as_posix()}"

# 输出 MIDI 文件夹：清理后的 MIDI 会保存到这里
output_midi_dir: "{OUTPUT_MIDI_DIR.as_posix()}"

# 目标 MIDI channel
# 这里使用正常音乐软件里的编号 1-16
target_channel: 1

# 是否递归查找子文件夹里的 .mid / .midi
recursive: false

# 如果 recursive=true，是否保留原始子文件夹结构
preserve_subfolders: true

# 输出文件已存在时是否覆盖
overwrite: true

# 是否写出 metadata csv
write_metadata: true

metadata_filename: "clean_midi_channel_metadata.csv"
'''

CONFIG_PATH.write_text(config_text, encoding="utf-8")

print(CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding="utf-8"))

D:\organ-amt-generalization\configs\clean_midi_channel.yaml
# clean_midi_channel.yaml

# 输入 MIDI 文件夹：只读取，不修改原文件
input_midi_dir: "D:/organ-amt-generalization/data/raw/organ/BWV545/midi"

# 输出 MIDI 文件夹：清理后的 MIDI 会保存到这里
output_midi_dir: "D:/organ-amt-generalization/data/processed/clearmidi"

# 目标 MIDI channel
# 这里使用正常音乐软件里的编号 1-16
target_channel: 1

# 是否递归查找子文件夹里的 .mid / .midi
recursive: false

# 如果 recursive=true，是否保留原始子文件夹结构
preserve_subfolders: true

# 输出文件已存在时是否覆盖
overwrite: true

# 是否写出 metadata csv
write_metadata: true

metadata_filename: "clean_midi_channel_metadata.csv"



## 3. 清理前检查 MIDI channel 分布

这一步不会修改文件，只统计输入文件夹中每个 MIDI 的 channel 分布。

In [4]:
def inspect_midi_channels(midi_path: Path) -> dict:
    mid = mido.MidiFile(str(midi_path))

    channel_msg_count = Counter()
    note_msg_count = Counter()
    track_note_count = Counter()
    track_channel_count = defaultdict(Counter)

    for ti, track in enumerate(mid.tracks):
        for msg in track:
            if hasattr(msg, "channel"):
                channel_msg_count[msg.channel + 1] += 1

            if msg.type in {"note_on", "note_off"}:
                track_note_count[ti] += 1
                if hasattr(msg, "channel"):
                    note_msg_count[msg.channel + 1] += 1
                    track_channel_count[ti][msg.channel + 1] += 1

    return {
        "path": str(midi_path),
        "file": midi_path.name,
        "num_tracks": len(mid.tracks),
        "length_sec": getattr(mid, "length", None),
        "channel_msg_count": dict(sorted(channel_msg_count.items())),
        "note_msg_count": dict(sorted(note_msg_count.items())),
        "track_note_count": dict(sorted(track_note_count.items())),
        "track_channel_count": {k: dict(v) for k, v in sorted(track_channel_count.items())},
    }


input_midi_files = sorted(list(INPUT_MIDI_DIR.glob("*.mid")) + list(INPUT_MIDI_DIR.glob("*.midi")))
print("input midi files:", len(input_midi_files))

before_records = [inspect_midi_channels(p) for p in input_midi_files]
pd.DataFrame(before_records)[["file", "num_tracks", "length_sec", "note_msg_count"]]

input midi files: 2


,file,num_tracks,length_sec,note_msg_count
0,BWV545i.mid,4,115.007579,"{1: 1102, 2: 570, 3: 340}"
1,BWV545ii.mid,5,229.026299,"{1: 894, 2: 864, 3: 990, 4: 508}"


## 4. 运行清理脚本

这一步会调用：

```powershell
python scripts\clean_midi_channel.py --config configs\clean_midi_channel.yaml
```

In [5]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--config",
    str(CONFIG_PATH),
]

print("Running command:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd=str(PROJECT_ROOT),
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code {result.returncode}")

Running command:
d:\anaconda3\envs\m1\python.exe D:\organ-amt-generalization\scripts\clean_midi_channel.py --config D:\organ-amt-generalization\configs\clean_midi_channel.yaml
STDOUT:
[INFO] Input MIDI dir: D:\organ-amt-generalization\data\raw\organ\BWV545\midi
[INFO] Output MIDI dir: D:\organ-amt-generalization\data\processed\clearmidi
[INFO] Found MIDI files: 2
[INFO] Target MIDI channel: 1
[INFO] Recursive: False
[INFO] Preserve subfolders: True
[INFO] Overwrite: True

[1/2] BWV545i.mid
[OK] Saved: D:\organ-amt-generalization\data\processed\clearmidi\BWV545i.mid
[INFO] changed_channel_messages=2027, note_messages=2012, tracks=4

[2/2] BWV545ii.mid
[OK] Saved: D:\organ-amt-generalization\data\processed\clearmidi\BWV545ii.mid
[INFO] changed_channel_messages=3276, note_messages=3256, tracks=5

[DONE] Processed: 2
[DONE] Output files recorded: 2
[DONE] Output dir: D:\organ-amt-generalization\data\processed\clearmidi

STDERR:



## 5. 清理后检查输出文件

确认输出文件都在：

```text
D:\organ-amt-generalization\data\processed\clearmidi
```

In [6]:
output_midi_files = sorted(list(OUTPUT_MIDI_DIR.glob("*.mid")) + list(OUTPUT_MIDI_DIR.glob("*.midi")))
print("output midi files:", len(output_midi_files))

for p in output_midi_files:
    print(p)

output midi files: 2
D:\organ-amt-generalization\data\processed\clearmidi\BWV545i.mid
D:\organ-amt-generalization\data\processed\clearmidi\BWV545ii.mid


## 6. 验证清理后是否全部为 Channel 1

如果 `note_msg_count` 只显示 `{1: ...}`，说明音符都已经转到 Channel 1。

In [7]:
after_records = [inspect_midi_channels(p) for p in output_midi_files]
df_after = pd.DataFrame(after_records)

df_after[["file", "num_tracks", "length_sec", "note_msg_count"]]

,file,num_tracks,length_sec,note_msg_count
0,BWV545i.mid,4,115.007579,{1: 2012}
1,BWV545ii.mid,5,229.026299,{1: 3256}


In [8]:
def all_notes_are_channel_1(record: dict) -> bool:
    note_counts = record["note_msg_count"]
    if len(note_counts) == 0:
        return True
    return set(note_counts.keys()) == {1}

check_rows = []
for r in after_records:
    check_rows.append({
        "file": r["file"],
        "all_notes_channel_1": all_notes_are_channel_1(r),
        "note_msg_count": r["note_msg_count"],
        "channel_msg_count": r["channel_msg_count"],
    })

df_check = pd.DataFrame(check_rows)
df_check

,file,all_notes_channel_1,note_msg_count,channel_msg_count
0,BWV545i.mid,True,{1: 2012},{1: 2027}
1,BWV545ii.mid,True,{1: 3256},{1: 3276}


## 7. 查看 metadata

脚本会生成：

```text
D:\organ-amt-generalization\data\processed\clearmidi\clean_midi_channel_metadata.csv
```

In [9]:
metadata_csv = OUTPUT_MIDI_DIR / "clean_midi_channel_metadata.csv"

if metadata_csv.exists():
    df_meta = pd.read_csv(metadata_csv)
    display(df_meta)
else:
    print("metadata csv not found:", metadata_csv)

,time,input_midi_path,output_midi_path,target_channel,num_tracks,num_channel_messages_changed,num_note_messages,status,error_message
0,2026-05-08T22:58:37,D:\organ-amt-generalization\data\raw\organ\BWV...,D:\organ-amt-generalization\data\processed\cle...,1,4,2027,2012,ok,NaN
1,2026-05-08T22:58:37,D:\organ-amt-generalization\data\raw\organ\BWV...,D:\organ-amt-generalization\data\processed\cle...,1,5,3276,3256,ok,NaN


## 8. 下一步用于渲染

后续 DAWDreamer 渲染时，把输入 MIDI 文件夹换成：

```text
D:\organ-amt-generalization\data\processed\clearmidi
```

也就是用清理后的 MIDI，而不是原始 MIDI。